# NimbusPay ticket triage — LoRA fine-tune of Qwen2.5-1.5B-Instruct

**Stack:** Hugging Face `transformers` + `peft` + `bitsandbytes`, plain `Trainer` with my own loss masking.
Why not Unsloth/TRL: the graders regenerate predictions with plain `model.generate` on base + PEFT adapter.
Training and predicting through exactly that code path removes one source of mismatch in the reproducibility
check, and writing the label mask myself (Part C) makes "loss on the right tokens" explicit and easy to verify.

**How to run**
* `MODE = "final"` (default): audit → augmentation → train the final config → dev score → test predictions →
  reproducibility check. Fits in < 3 h on a fresh T4.
* `MODE = "experiment"`: set `RUN = "E1_raw"` etc. and run once per experiment. Every run saves adapter,
  dev predictions and metrics to Drive, so the results table (Part D) is rebuilt from saved logs.

Put these files in `MyDrive/nimbuspay_triage/`: `train.jsonl`, `dev.jsonl`, `test_inputs.jsonl`, `SCHEMA.md`,
`score.py`, `rules.py`, `audit.py`, `augment.py`.

## 0. Setup

In [ ]:
MODE = "final"          # "final" or "experiment"
RUN  = "E2_clean"       # used when MODE == "experiment"
FINAL_RUN = "E5_aug"    # best config from Part D (update after experiments)
REUSE_ADAPTER = True   # final mode: reuse the FINAL_RUN adapter on Drive if it exists (False = retrain from scratch)


In [ ]:
!pip -q install -U peft bitsandbytes
import torch, transformers, peft, bitsandbytes, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
DRIVE = "/content/drive/MyDrive/nimbuspay_triage"
WORK = "/content/work"; os.makedirs(f"{WORK}/data", exist_ok=True); os.makedirs(f"{DRIVE}/runs", exist_ok=True)
for f in ["train.jsonl", "dev.jsonl", "test_inputs.jsonl"]:
    shutil.copy(f"{DRIVE}/{f}", f"{WORK}/data/{f}")
for f in ["SCHEMA.md", "score.py", "rules.py", "audit.py", "augment.py"]:
    shutil.copy(f"{DRIVE}/{f}", f"{WORK}/{f}")
os.chdir(WORK); sys.path.insert(0, WORK)
print(sorted(os.listdir(WORK)), sorted(os.listdir("data")))

In [ ]:
import json, re, time, gc, random, collections
import numpy as np, torch, transformers, os, shutil, sys
import rules as R, score as S

def load(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def save_jsonl(rows, p):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM = "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
KEYS = S.KEYS
SEED = 42

## Part A — Data audit
`audit.py` + `rules.py` (in the repo) do the work; this section re-runs them and shows the evidence.
`rules.py` is a rule extractor written from SCHEMA.md. It is used **only** to audit training labels and to
label augmentation data — never at inference.

In [ ]:
# 1) the rule extractor reproduces every non-category dev label -> it is a trustworthy reference for auditing
import audit as A
dev = load("data/dev.jsonl")
dis = collections.Counter()
for r in dev:
    g = json.loads(r["messages"][2]["content"]); rec, _, body = R.split_ticket(r["messages"][1]["content"])
    for k, (lab, ref) in A.residual_disagreements(g, body, rec).items(): dis[k] += 1
    if g["priority"] != R.priority(g["category"], g["amount_paise"], body): dis["priority"] += 1
    if g["needs_human"] != R.needs_human(g["priority"], body): dis["needs_human"] += 1
print(f"dev rows: {len(dev)}   rule-vs-gold disagreements per field: {dict(dis) or 'none'}")

In [ ]:
# 2) run the audit: writes data/train_clean.jsonl and cleaning_log.csv, prints counts
!python audit.py --train data/train.jsonl --dev data/dev.jsonl --test data/test_inputs.jsonl --out data/train_clean.jsonl --log cleaning_log.csv

In [ ]:
# 3) one concrete example of each issue type (before -> after)
import csv
raw = {r["id"]: r for r in load("data/train.jsonl")}
clean = {r["id"]: r for r in load("data/train_clean.jsonl")}
log = list(csv.DictReader(open("cleaning_log.csv", encoding="utf-8")))
shown = set()
for row in log:
    kind = row["reason"].split(":")[0].split(";")[0]
    if kind in shown: continue
    shown.add(kind); r = raw[row["id"]]
    print(f"=== {kind} ({row['action']}) {row['id']}")
    print("ticket:", r["messages"][1]["content"].split("\n\n", 1)[1][:220].replace("\n", " | "))
    print("label :", r["messages"][2]["content"][:220])
    if row["id"] in clean: print("fixed :", clean[row["id"]]["messages"][2]["content"])
    print("reason:", row["reason"], "\n")

In [ ]:
# 4) category audit: out-of-fold TF-IDF + logistic regression (confident-learning style).
# Rows where a model trained on the other folds confidently disagrees with the label were read by hand.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline, make_union
rows = load("data/train_clean.jsonl")
X = [re.sub(r"\d", "0", R.unquoted(R.split_ticket(r["messages"][1]["content"])[2]).lower()) for r in rows]
y = np.array([json.loads(r["messages"][2]["content"])["category"] for r in rows])
clf = make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                               TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
                    LogisticRegression(C=10, max_iter=3000))
P = cross_val_predict(clf, X, y, cv=5, method="predict_proba"); cls = np.array(sorted(set(y))); pred = cls[P.argmax(1)]
print(f"out-of-fold accuracy {np.mean(pred == y):.3f}; most confident disagreements:")
for i in sorted(np.where(pred != y)[0], key=lambda i: -P[i].max())[:12]:
    print(f"  {rows[i]['id']} label={y[i]:15} model={pred[i]:15} p={P[i].max():.2f} | {X[i][:110]!r}")
print("Verdict after reading them: the labels follow SCHEMA (e.g. 'a payment I don't recognise' is fraud);")
print("the classifier is wrong, not the label. No category relabelling.")

In [ ]:
# 5) augmentation for experiment E5 (see augment.py docstring)
!python augment.py --train data/train_clean.jsonl --out data/train_aug.jsonl --n 800 --seed 0
aug = load("data/train_aug.jsonl")
print(aug[3]["messages"][1]["content"], "\n->", aug[3]["messages"][2]["content"])

## Part C — Training pipeline

**Chat template.** Qwen2.5 has a native system role, so the system and user messages go through
`tokenizer.apply_chat_template` unchanged — exactly what the graders do at inference.

**Loss on the right tokens.** Each example is tokenised as `prompt_ids + completion_ids`, where
`prompt = apply_chat_template([system, user], add_generation_prompt=True)` (ends with `<|im_start|>assistant\n`)
and `completion = label JSON + <|im_end|>`. Labels are `-100` on every prompt token, so loss is only on the JSON
and the end-of-turn token (which teaches the model to stop: `<|im_end|>` is an EOS id in Qwen's generation config).
Tokenising the two parts separately guarantees the boundary is exact; the cell below asserts that the concatenation
equals the template's own rendering of the full conversation.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
END = "<|im_end|>"

def prompt_text(msgs): return tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)

def encode(row):
    p = tok(prompt_text(row["messages"]), add_special_tokens=False)["input_ids"]
    c = tok(row["messages"][2]["content"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + c, "attention_mask": [1] * (len(p) + len(c)), "labels": [-100] * len(p) + c}

r0 = load("data/train_clean.jsonl")[0]
full = tok.apply_chat_template(r0["messages"], tokenize=False)
assert full == prompt_text(r0["messages"]) + r0["messages"][2]["content"] + END + "\n", "template boundary mismatch"
e = encode(r0)
print(full)
print("--- tokens that receive loss:")
print(repr(tok.decode([t for t, l in zip(e["input_ids"], e["labels"]) if l != -100])))

In [ ]:
# sequence length from evidence: never truncate (truncation would cut the label, which sits at the end)
import matplotlib.pyplot as plt
lens = {name: [len(encode(r)["input_ids"]) for r in load(f"data/{name}.jsonl")]
        for name in ["train", "train_clean", "train_aug"]}
for k, v in lens.items(): print(f"{k:12} p50={np.percentile(v,50):.0f} p95={np.percentile(v,95):.0f} p99={np.percentile(v,99):.0f} max={max(v)}")
comp = [len(tok(r["messages"][2]["content"] + END)["input_ids"]) for r in load("data/train_clean.jsonl")]
print(f"label length: max {max(comp)} tokens  (max_new_tokens=200 leaves ample room)")
MAX_LEN = int(np.ceil(max(max(v) for v in lens.values()) / 64) * 64)
print("MAX_LEN =", MAX_LEN)
plt.hist(lens["train_clean"], bins=60); plt.axvline(MAX_LEN, c="r"); plt.xlabel("tokens"); plt.title("train_clean lengths"); plt.show()

In [ ]:
import transformers
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset
from packaging import version
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)   # T4 has no bf16

def load_base(four_bit):
    kw = {DTYPE_KW: torch.float16, "device_map": {"": 0}, "attn_implementation": "sdpa"}
    if four_bit: kw["quantization_config"] = BNB
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)

@torch.no_grad()
def generate(model, rows, system=None, bs=1):
    # Greedy, max_new_tokens=200, chat template, strip only. bs=1 is exactly the graders' setting.
    model.eval(); tok.padding_side = "left"; out = []
    for i in range(0, len(rows), bs):
        chunk = rows[i:i + bs]
        msgs = [[{"role": "system", "content": system or r["messages"][0]["content"]}, r["messages"][1]] for r in chunk]
        enc = tok([prompt_text(m) for m in msgs], return_tensors="pt", padding=True, add_special_tokens=False).to(0)
        gen = model.generate(**enc, max_new_tokens=200, do_sample=False, temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
        for r, g in zip(chunk, gen[:, enc["input_ids"].shape[1]:]):
            out.append({"id": r["id"], "output": tok.decode(g, skip_special_tokens=True).strip()})
    return out

def evaluate(preds, gold_rows):
    res = S.score(gold_rows, preds)
    a = res["all"]
    print(f"valid {a['json_valid']:.1%}  exact {a['exact_match']:.1%}  fields {a['mean_field_acc']:.1%}")
    print("  " + "  ".join(f"{k}={v:.1%}" for k, v in a["fields"].items()))
    return res

In [ ]:
CONFIGS = {   # every experiment changes ONE thing relative to E2_clean
    "E1_raw":     dict(data="train",      four_bit=True,  r=16, epochs=3, lr=2e-4),
    "E2_clean":   dict(data="train_clean",four_bit=True,  r=16, epochs=3, lr=2e-4),   # reference
    "E3_lora16":  dict(data="train_clean",four_bit=False, r=16, epochs=3, lr=2e-4),
    "E4_1epoch":  dict(data="train_clean",four_bit=True,  r=16, epochs=1, lr=2e-4),
    "E5_aug":     dict(data="train_clean+train_aug", four_bit=True, r=16, epochs=3, lr=2e-4),
    # E6 changes one thing relative to E5: key order of the training labels. priority depends on category, amount
    # and a legal threat, and needs_human depends on priority, so emitting them LAST lets the model condition on the
    # amount it has already written. Scoring ignores key order; SCHEMA only requires exactly the 8 keys.
    "E6_order":   dict(data="train_clean+train_aug", four_bit=True, r=16, epochs=3, lr=2e-4, order="derived_last"),
}
DERIVED_LAST = ["category", "amount_paise", "txn_date", "txn_id", "channel", "language", "priority", "needs_human"]

import torch.nn.functional as F

class LabelOnlyLossTrainer(Trainer):
    # Same loss as the default causal-LM loss (mean token cross-entropy over label tokens), but the 152k-way
    # vocabulary projection runs only at the ~85 positions that carry a label, not at all ~2.2k positions of a
    # long quoted-thread ticket. Measured locally: the default path needs ~18 GB for a batch of 4 such tickets
    # (fp32 logits 4 x 2240 x 151936), which would OOM a 15 GB T4.
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.model_accepts_loss_kwargs = False      # -> Trainer divides our per-micro-batch mean by grad-accum steps

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        core = model.get_base_model()                # Qwen2ForCausalLM with LoRA layers injected
        hidden = core.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
        target = labels[:, 1:]                       # hidden state at position t predicts token t+1
        keep = target != -100
        logits = core.lm_head(hidden[:, :-1][keep].to(core.lm_head.weight.dtype)).float()
        loss = F.cross_entropy(logits, target[keep])
        return (loss, None) if return_outputs else loss

def train_rows(spec, order=None):
    rows = []
    for part in spec.split("+"): rows += load(f"data/{part}.jsonl")
    # raw data contains unparseable labels; we still train on them verbatim for E1 (that IS the raw condition)
    if order == "derived_last":
        for r in rows:
            lab = json.loads(r["messages"][2]["content"])
            r["messages"][2]["content"] = json.dumps({k: lab[k] for k in DERIVED_LAST}, ensure_ascii=False)
    return rows

def train_and_eval(name, cfg, eval_rows, save_to):
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = load_base(cfg["four_bit"])
    if cfg["four_bit"]:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    lcfg = LoraConfig(r=cfg["r"], lora_alpha=2 * cfg["r"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lcfg)
    for p in model.parameters():          # fp16 AMP needs fp32 master weights for the trainable params
        if p.requires_grad: p.data = p.data.float()
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.1f}M (limit 50M)"); assert n_train <= 50e6

    rows = train_rows(cfg["data"], cfg.get("order"))
    ds = Dataset.from_list([encode(r) for r in rows]).shuffle(seed=SEED)
    assert max(len(x) for x in ds["input_ids"]) <= MAX_LEN
    bs, ga = 4, 4   # effective batch 16; bs 4 keeps the worst case (4 x ~2.2k-token quoted threads) inside T4 memory
    steps = int(np.ceil(len(ds) / (bs * ga)) * cfg["epochs"])
    args = TrainingArguments(output_dir=f"/content/ckpt/{name}", per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
                             num_train_epochs=cfg["epochs"], learning_rate=cfg["lr"], lr_scheduler_type="cosine",
                             warmup_steps=max(1, int(0.05 * steps)), weight_decay=0.0, max_grad_norm=1.0, fp16=True,
                             logging_steps=10, save_strategy="no", report_to="none", seed=SEED, optim="adamw_torch",
                             gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    tok.padding_side = "right"
    trainer = LabelOnlyLossTrainer(model=model, args=args, train_dataset=ds,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    t0 = time.time(); trainer.train(); minutes = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    loss_log = [l for l in trainer.state.log_history if "loss" in l]
    os.makedirs(save_to, exist_ok=True)
    model.save_pretrained(f"{save_to}/adapter")
    # Evaluate exactly the way the graders will: fresh base model + saved adapter.
    # (prepare_model_for_kbit_training upcasts norms/embeddings to fp32 during training; a fresh load keeps
    #  them fp16, and greedy outputs can differ. Local test showed in-memory vs reloaded predictions differ.)
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    model = PeftModel.from_pretrained(load_base(cfg["four_bit"]), f"{save_to}/adapter").eval()
    preds = generate(model, eval_rows, bs=1)
    res = evaluate(preds, eval_rows)
    save_jsonl(preds, f"{save_to}/dev_preds.jsonl")
    summary = dict(name=name, **cfg, train_rows=len(rows), trainable_M=round(n_train / 1e6, 1), steps=steps,
                   train_minutes=round(minutes, 1), peak_gpu_gb=round(peak_gb, 2),
                   exact=res["all"]["exact_match"], fields=res["all"]["mean_field_acc"], scores=res, loss_log=loss_log)
    json.dump(summary, open(f"{save_to}/result.json", "w"), indent=1)
    print(f"[{name}] {minutes:.1f} min, peak {peak_gb:.2f} GB")
    return model, preds, summary

## Part B — Baselines (base model, no fine-tuning)
B1: fixed short system prompt. B2: the SCHEMA rules pasted into the system prompt (allowed here only, as a reference).
Batched generation (bs=8) is used here only, to save GPU time.

In [ ]:
if MODE == "experiment" and RUN in ("B1", "B2"):
    base = load_base(four_bit=True)
    schema = open("SCHEMA.md", encoding="utf-8").read()
    rules_prompt = SYSTEM + " Output only the JSON object.\n\n" + schema[schema.index("## Output"):]
    for name, system in [("B1", None), ("B2", rules_prompt)]:
        t0 = time.time(); preds = generate(base, dev, system=system, bs=8); res = evaluate(preds, dev)
        d = f"{DRIVE}/runs/{name}"; os.makedirs(d, exist_ok=True); save_jsonl(preds, f"{d}/dev_preds.jsonl")
        json.dump(dict(name=name, exact=res["all"]["exact_match"], fields=res["all"]["mean_field_acc"], scores=res,
                       eval_minutes=round((time.time() - t0) / 60, 1), peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2)),
                  open(f"{d}/result.json", "w"), indent=1)
        print(name, "sample output:", preds[0]["output"][:300], "\n")

## Part D — Controlled experiments

In [ ]:
if MODE == "experiment" and RUN in CONFIGS:
    model, preds, summary = train_and_eval(RUN, CONFIGS[RUN], dev, f"{DRIVE}/runs/{RUN}")

In [ ]:
# results table, rebuilt from the saved logs of every run
import pandas as pd
rows_ = []
for name in ["B1", "B2"] + list(CONFIGS):
    p = f"{DRIVE}/runs/{name}/result.json"
    if os.path.exists(p):
        r = json.load(open(p))
        rows_.append({"run": name, "data": r.get("data", "-"), "4-bit": r.get("four_bit", True), "r": r.get("r", "-"),
                      "epochs": r.get("epochs", "-"), "key order": r.get("order") or "schema", "dev exact": f"{r['exact']:.1%}", "dev fields": f"{r['fields']:.1%}",
                      "peak GPU GB": r.get("peak_gpu_gb"), "train min": r.get("train_minutes", "-")})
pd.DataFrame(rows_)

## Final model: train, score on dev, predict test

In [ ]:
if MODE == "final":
    run_dir = f"{DRIVE}/runs/{FINAL_RUN}"
    if REUSE_ADAPTER and os.path.exists(f"{run_dir}/adapter/adapter_model.safetensors"):
        # adapter already trained by the experiment run of the same config: load it instead of retraining
        model = PeftModel.from_pretrained(load_base(CONFIGS[FINAL_RUN]["four_bit"]), f"{run_dir}/adapter").eval()
        print("reusing", run_dir, json.load(open(f"{run_dir}/result.json"))["exact"])
    else:
        model, dev_preds, summary = train_and_eval(FINAL_RUN, CONFIGS[FINAL_RUN], dev, run_dir)
    # `model` is a freshly loaded base + adapter (graders' path)
    shutil.copytree(f"{DRIVE}/runs/{FINAL_RUN}/adapter", "/content/adapter", dirs_exist_ok=True)
    test = load("data/test_inputs.jsonl")
    t0 = time.time(); test_preds = generate(model, test, bs=1)
    save_jsonl(test_preds, "/content/predictions_test.jsonl"); save_jsonl(test_preds, f"{DRIVE}/predictions_test.jsonl")
    shutil.copytree("/content/adapter", f"{DRIVE}/adapter", dirs_exist_ok=True)
    print(f"test predictions: {len(test_preds)} rows in {(time.time()-t0)/60:.1f} min;",
          f"valid JSON {np.mean([S.parse(p['output']) is not None for p in test_preds]):.1%}")

In [ ]:
# Reproducibility check, the way the graders do it: fresh base model + adapter from disk, plain generate,
# regenerate a random sample of test rows and compare byte-for-byte.
if MODE == "final":
    del model; gc.collect(); torch.cuda.empty_cache()
    fresh = PeftModel.from_pretrained(load_base(CONFIGS[FINAL_RUN]["four_bit"]), "/content/adapter")
    sample = random.Random(0).sample(test, 25)
    again = {p["id"]: p["output"] for p in generate(fresh, sample, bs=1)}
    mism = [p["id"] for p in test_preds if p["id"] in again and again[p["id"]] != p["output"]]
    print(f"regenerated {len(sample)} rows, mismatches: {len(mism)} {mism}")

## Part E — Error analysis (best model on dev)

In [ ]:
if os.path.exists(f"{DRIVE}/runs/{FINAL_RUN}/dev_preds.jsonl"):
    best = FINAL_RUN
    preds = {p["id"]: p["output"] for p in load(f"{DRIVE}/runs/{best}/dev_preds.jsonl")}
    res = S.score(dev, [{"id": k, "output": v} for k, v in preds.items()])
    print(pd.DataFrame({g: {"n": r["n"], "exact": r["exact_match"], **r["fields"]} for g, r in res.items()}).T
          .sort_values("exact").to_string(float_format=lambda x: f"{x:.2f}"))
else:
    print("Part E skipped: run FINAL_RUN first (it needs runs/" + FINAL_RUN + "/dev_preds.jsonl)")


In [ ]:
if "preds" in globals() and isinstance(preds, dict):
    fails, wrong_by_field = [], collections.Counter()
    for r in dev:
        gold = json.loads(r["messages"][2]["content"]); p = S.parse(preds.get(r["id"])) or {}
        wrong = [k for k in KEYS if not S.same(p.get(k, "<missing>"), gold[k])]
        if wrong:
            fails.append((r, gold, p, wrong)); wrong_by_field.update(wrong)
    print(len(fails), "failures; wrong fields:", wrong_by_field.most_common())
    cat_conf = collections.Counter((g["category"], p.get("category")) for _, g, p, w in fails if "category" in w)
    print("category confusions (gold -> pred):", cat_conf.most_common())
    for r, gold, p, wrong in fails[:20]:
        print(f"\n--- {r['id']}  wrong: {wrong}")
        print(r["messages"][1]["content"][:600])
        for k in wrong: print(f"   {k}: gold={gold[k]!r}  pred={p.get(k, '<missing>')!r}")
else:
    print("Part E skipped")
